# E11 band pipeline: Kaggle run

Runs `scripts/run_band_pipeline.py` exactly as pre-registered (`results/notes/experiment_log.md`, E11):
checks -> smoke run -> dev search -> one test run -> packaged outputs.

**Inputs:** two private datasets made from `kaggle/make_bundle.sh`: one holding `trr_code.zip`, one holding `trr_data.zip`.

**Settings:** Internet **on** (pip install, embedding model download). Accelerator **GPU T4** for the dense baselines
(Parth's final method, dense doc-first). Without a GPU the notebook runs the lexical part only.

**Output:** `/kaggle/working/e11_outputs.zip`. Download it and unzip it in the repository root.

In [ ]:
import glob, os, shutil, subprocess, time
from pathlib import Path

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under /kaggle/input: are both datasets added?")
    return Path(hits[0])

code_root = find("configs/default.yaml").parents[1]
data_root = find("data/raw/legalbenchrag/corpus").parents[3]

REPO = Path("/kaggle/working/trr")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(code_root, REPO)
subprocess.run(["chmod", "-R", "u+w", str(REPO)], check=True)        # inputs are read-only
(REPO / "data/raw").mkdir(parents=True, exist_ok=True)
os.symlink(data_root / "data/raw/legalbenchrag", REPO / "data/raw/legalbenchrag")
(REPO / "data/cache").mkdir(parents=True, exist_ok=True)
for f in (data_root / "data/cache").glob("*.npy"):
    shutil.copy(f, REPO / "data/cache" / f.name)
(REPO / "results/logs").mkdir(parents=True, exist_ok=True)
print("code at", REPO, "| git", (REPO / "GIT_HASH").read_text().strip())
print("data from", data_root)

In [ ]:
def run(cmd, log=None):
    """Run a shell command in the repo, stream its output, keep a log, stop the notebook on failure."""
    print("$", cmd, flush=True)
    t0, lines = time.time(), []
    with subprocess.Popen(cmd, shell=True, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
            lines.append(line)
    if log:
        (REPO / "results/logs" / log).write_text("".join(lines))
    print(f"[{time.time() - t0:.0f} s]", flush=True)
    if p.returncode:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

run("pip install -q -e '.[dev]'")

## 1. Checks: unit tests, and the dev/test split must be identical to the local one

In [ ]:
run("python -m pytest -q", log="e11_pytest.log")

import pandas as pd
local = pd.read_csv(REPO / "results/tables/query_split.csv")
run("python scripts/make_query_split.py")
here = pd.read_csv(REPO / "results/tables/query_split.csv")
assert local.equals(here), "dev/test split differs from the local one: stop and investigate"
print("split identical:", (here.side == "dev").sum(), "dev /", (here.side == "test").sum(), "test queries")

## 2. Smoke run (12 documents per subset): catches errors in a minute

In [ ]:
run("python scripts/run_band_pipeline.py --stage dev --smoke 12", log="e11_smoke_dev.log")
run("python scripts/run_band_pipeline.py --stage test --smoke 12", log="e11_smoke_test.log")

## 3. Dev search (dev queries only): picks the setting

In [ ]:
run("python scripts/run_band_pipeline.py --stage dev", log="e11_dev.log")
print((REPO / "results/tables/band_dev.md").read_text())

## 4. Test run, once

In [ ]:
import torch
dense = torch.cuda.is_available()
print("GPU available:", dense, "-> dense baselines", "ON" if dense else "OFF (lexical part only)")
run("python scripts/run_band_pipeline.py --stage test" + (" --dense-baselines" if dense else ""), log="e11_test.log")
print((REPO / "results/tables/band_test.md").read_text())

## 5. Package the outputs

In [ ]:
import zipfile
out = Path("/kaggle/working/e11_outputs.zip")
patterns = ["results/tables/band_*", "results/runs/*_test*.jsonl", "results/logs/e11_*.log",
            "data/cache/chunk_emb_*.npy"]      # chunk embeddings: reused by the dense pipelines later
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in patterns:
        for f in sorted(REPO.glob(pattern)):
            z.write(f, f.relative_to(REPO))
            print(f.relative_to(REPO))
print(out, f"{out.stat().st_size / 1e6:.0f} MB")